# Notebook 2 - PCA / SVD

**DATA5322 - Statistical Machine Learning II**  
Seattle University, Spring 2026  
Team: Ruman Sidhu, Paul Skentzos, Hamda Hassan

---

## Overview

This notebook applies Principal Component Analysis (PCA) and Singular Value Decomposition (SVD)
to the preprocessed BC-TCGA breast cancer gene expression matrix produced in Notebook 1.

The preprocessed matrix `data/X_preprocessed.npy` contains **529 tumor samples x 5,000 genes**,
already filtered to the top-variance genes and standardized to zero mean and unit variance.

### Goals
1. Run PCA and SVD on the expression matrix
2. Produce a scree plot and discuss the variance structure
3. Interpret the U and V* matrices from SVD in the context of gene expression
4. Plot tumor samples in PC1/PC2 space and look for inherent structure
5. Identify the top contributing genes per principal component
6. Decide how many PCs to retain for downstream clustering (Notebooks 4 and 5)

---
## 1. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from pathlib import Path

# Match Paul's setup from Notebook 1
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

DATA_DIR  = Path('../data')
PLOTS_DIR = Path('../plots')
PLOTS_DIR.mkdir(exist_ok=True)

print('Setup complete.')

---
## 2. Load Preprocessed Data

We load the output from Notebook 1:
- `X_preprocessed.npy`: the standardized 529 x 5,000 expression matrix
- `top_genes.csv`: names of the 5,000 selected genes
- `sample_ids.csv`: TCGA sample identifiers

In [ ]:
X          = np.load(DATA_DIR / 'X_preprocessed.npy')
gene_names = pd.read_csv(DATA_DIR / 'top_genes.csv')['Gene'].tolist()
sample_ids = pd.read_csv(DATA_DIR / 'sample_ids.csv')['SampleID'].tolist()

print(f'Expression matrix shape : {X.shape}  (samples x genes)')
print(f'Number of gene names    : {len(gene_names)}')
print(f'Number of sample IDs    : {len(sample_ids)}')
print(f'Matrix mean             : {X.mean():.6f}  (should be ~0)')
print(f'Matrix std              : {X.std():.6f}   (should be ~1)')

---
## 3. Run PCA

We fit PCA on the full matrix. Since the data is already standardized from Notebook 1,
no further scaling is needed. We retain all components initially so we can examine
the full variance spectrum before deciding how many to keep.

In [ ]:
# Fit PCA - retain all components to see the full variance spectrum
pca   = PCA()
X_pca = pca.fit_transform(X)

explained_var  = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

print(f'PCA fit complete.')
print(f'Total components        : {len(explained_var)}')
print(f'Variance in PC1         : {explained_var[0]*100:.2f}%')
print(f'Variance in PC2         : {explained_var[1]*100:.2f}%')
print(f'Variance in PC3         : {explained_var[2]*100:.2f}%')
print(f'Cumulative (top 10 PCs) : {cumulative_var[9]*100:.2f}%')
print(f'Cumulative (top 50 PCs) : {cumulative_var[49]*100:.2f}%')

---
## 4. Scree Plot and Proportion of Variance Explained

The scree plot shows how much variance each principal component captures.
In gene expression data we expect a sharp initial drop followed by a long flat tail.
This shape tells us that a small number of components capture the dominant biological
signals (like broad subtype distinctions), while the remaining components capture
progressively less meaningful variation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: individual variance per PC (scree plot)
axes[0].plot(range(1, 51), explained_var[:50] * 100, 'o-',
             color='#7B68EE', markersize=5, linewidth=1.5)
axes[0].axhline(y=1, color='#DA70D6', linestyle='--', alpha=0.7, label='1% threshold')
axes[0].set_xlabel('Principal Component', fontsize=12)
axes[0].set_ylabel('Variance Explained (%)', fontsize=12)
axes[0].set_title('Scree Plot (Top 50 PCs)', fontsize=13)
axes[0].legend()

# Right: cumulative variance
axes[1].plot(range(1, 101), cumulative_var[:100] * 100, 'o-',
             color='#00CED1', markersize=3, linewidth=1.5)
axes[1].axhline(y=50, color='#7B68EE', linestyle='--', alpha=0.7, label='50% variance')
axes[1].axhline(y=80, color='#FF69B4',  linestyle='--', alpha=0.7, label='80% variance')
axes[1].axhline(y=90, color='#DA70D6',    linestyle='--', alpha=0.7, label='90% variance')
axes[1].set_xlabel('Number of Principal Components', fontsize=12)
axes[1].set_ylabel('Cumulative Variance Explained (%)', fontsize=12)
axes[1].set_title('Cumulative Variance Explained (Top 100 PCs)', fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'scree_plot.png', dpi=150, bbox_inches='tight')
plt.show()

# Print key thresholds
print('PCs needed to reach variance thresholds:')
for threshold in [0.50, 0.80, 0.90]:
    n = np.argmax(cumulative_var >= threshold) + 1
    print(f'  {threshold*100:.0f}%  ->  {n} PCs')

### Discussion: What does the scree plot shape tell us?

The scree plot shows a steep initial drop where the first few components capture a
disproportionately large share of the variance, followed by a gradual flattening into
an elbow shape. This is the classic pattern for structured biological data.

In gene expression terms, the early PCs capture the dominant axes of biological
variation across all 529 tumor samples. These likely correspond to major sources of
differences between tumors such as hormone receptor status, proliferation rate, or
HER2 amplification, which are exactly the signals that define breast cancer molecular subtypes.

The long flat tail after the elbow represents thousands of genes contributing small,
relatively noisy amounts of variation. Including too many of these components in
downstream clustering would add noise without meaningful biological signal.

The cumulative plot helps us decide practically how many PCs to retain for clustering.

---
## 5. SVD Decomposition and Interpreting U and V*

PCA is mathematically equivalent to SVD. We perform SVD explicitly here to
demonstrate the decomposition and interpret each matrix in the context of gene expression.

For our matrix X (529 samples x 5000 genes):
- **U** (samples x components): each row is a sample coordinate in the new low-dimensional space
- **S** (singular values): the importance/magnitude of each component
- **V*** (components x genes): each row is a PC, each value is a gene loading

In [ ]:
# Run SVD directly using numpy
U, S, Vt = np.linalg.svd(X, full_matrices=False)

print('SVD decomposition shapes:')
print(f'  U  (samples x components) : {U.shape}')
print(f'  S  (singular values)       : {S.shape}')
print(f'  V* (components x genes)    : {Vt.shape}')
print(f'\nFirst 5 singular values : {S[:5].round(2)}')
print(f'Variance from S[0]      : {(S[0]**2 / (S**2).sum() * 100):.2f}%')
print(f'Variance from S[1]      : {(S[1]**2 / (S**2).sum() * 100):.2f}%')

In [ ]:
# Singular value decay plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, 51), S[:50], 'o-', color='#7B68EE', markersize=5)
axes[0].fill_between(range(1, 51), S[:50], alpha=0.15, color='#7B68EE')
axes[0].set_xlabel('Component Index', fontsize=12)
axes[0].set_ylabel('Singular Value', fontsize=12)
axes[0].set_title('Singular Value Decay (Top 50 Components)', fontsize=13)

s_var = (S**2) / (S**2).sum()
axes[1].plot(range(1, 51), s_var[:50] * 100, 'o-', color='#00CED1', markersize=5)
axes[1].set_xlabel('Component Index', fontsize=12)
axes[1].set_ylabel('Variance Explained (%)', fontsize=12)
axes[1].set_title('Variance Explained via Singular Values', fontsize=13)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'svd_singular_values.png', dpi=150, bbox_inches='tight')
plt.show()

### Discussion: Interpreting U and V* in the context of gene expression

**The U matrix (sample coordinates):**
Each row of U corresponds to one of our 529 breast cancer tumor samples.
The values in that row tell us where the sample sits in the new low-dimensional
space defined by the principal components. Samples that are close together in U-space
have similar gene expression profiles. When we plot tumors using the first two columns
of U, we are directly visualizing this similarity structure and asking whether natural
biological groups emerge without any label information.

**The V* matrix (gene loadings):**
Each row of V* corresponds to one principal component. Each column corresponds to
one of our 5,000 genes. The value at position [i, j] tells us how strongly gene j
contributes to component i. Genes with large absolute loadings on PC1 are the ones
driving the biggest axis of variation across all tumors. In breast cancer biology
these tend to be genes related to estrogen receptor signaling, cell proliferation,
or HER2 expression, which are the markers that define molecular subtypes.

**The singular values S:**
S tells us the magnitude of each component. A steeply decaying S confirms genuine
low-rank structure in the data, meaning a small number of biological processes
explain most of the variation between tumors.

---
## 6. Top Gene Loadings Per Principal Component

The loading values in V* tell us which genes are most responsible for each PC.
Genes with large positive loadings push samples in the positive direction along
that PC, while genes with large negative loadings push them in the opposite direction.

In [ ]:
# Build loadings DataFrame from PCA components (equivalent to V* from SVD)
n_show   = 4
loadings = pd.DataFrame(
    pca.components_[:n_show, :],
    columns=gene_names,
    index=[f'PC{i+1}' for i in range(n_show)]
)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, ax in enumerate(axes):
    top_idx = loadings.iloc[i].abs().sort_values(ascending=False).index[:20]
    vals    = loadings.iloc[i][top_idx]
    colors  = ['#7B68EE' if v > 0 else '#DA70D6' for v in vals]

    ax.barh(range(20), vals.values[::-1], color=colors[::-1])
    ax.set_yticks(range(20))
    ax.set_yticklabels(top_idx[::-1], fontsize=8)
    ax.set_xlabel('Loading Value', fontsize=10)
    ax.set_title(
        f'PC{i+1} - Top 20 Gene Loadings  '
        f'(Variance explained: {explained_var[i]*100:.1f}%)',
        fontsize=11
    )
    ax.axvline(x=0, color='black', linewidth=0.8)

plt.suptitle(
    'Top Gene Loadings per Principal Component\n(Blue = positive contribution, Red = negative)',
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'gene_loadings.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the top contributing gene per PC
print('Top contributing gene per PC:')
for i in range(4):
    top = loadings.iloc[i].abs().idxmax()
    val = loadings.iloc[i][top]
    print(f'  PC{i+1}: {top}  (loading = {val:.4f})')

---
## 7. Plotting Samples in PC1 / PC2 Space

We project all 529 tumor samples into the 2D space defined by the first two
principal components. If biological subtype structure exists in the data, we
expect natural clusters or groupings to emerge here without any labels.
This is the first visual test of our central hypothesis.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ax.scatter(
    X_pca[:, 0], X_pca[:, 1],
    alpha=0.65, s=25, c='#7B68EE', edgecolors='none'
)

ax.set_xlabel(f'PC1  ({explained_var[0]*100:.1f}% variance explained)', fontsize=12)
ax.set_ylabel(f'PC2  ({explained_var[1]*100:.1f}% variance explained)', fontsize=12)
ax.set_title(
    'BC-TCGA Breast Cancer Tumor Samples in PC1 / PC2 Space\n'
    f'(n = {X.shape[0]} samples, no labels)',
    fontsize=13
)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3, linewidth=0.8)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.3, linewidth=0.8)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'biplot_pc1_pc2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compare PC space vs two original features side by side
X_df = pd.DataFrame(X, columns=gene_names)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PC space
axes[0].scatter(X_pca[:, 0], X_pca[:, 1],
                alpha=0.6, s=20, c='#7B68EE', edgecolors='none')
axes[0].set_xlabel(f'PC1 ({explained_var[0]*100:.1f}% var)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({explained_var[1]*100:.1f}% var)', fontsize=11)
axes[0].set_title('Samples in PC1 / PC2 Space\n(captures dominant biological variation)', fontsize=11)

# Two original genes
gene1, gene2 = gene_names[0], gene_names[1]
axes[1].scatter(X_df[gene1], X_df[gene2],
                alpha=0.6, s=20, c='#00CED1', edgecolors='none')
axes[1].set_xlabel(f'{gene1}  (standardized expression)', fontsize=11)
axes[1].set_ylabel(f'{gene2}  (standardized expression)', fontsize=11)
axes[1].set_title('Samples Plotted on Two Original Genes\n(arbitrary, hard to interpret)', fontsize=11)

plt.suptitle('PC Space vs Original Feature Space', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'pc_vs_original.png', dpi=150, bbox_inches='tight')
plt.show()

### Discussion: What structure exists in PC space vs original features?

**PC space:**
Plotting in PC1/PC2 space reveals whether any natural groupings exist in the tumor data.
PC1 and PC2 together capture the two largest axes of variation across all 529 samples.
Any visible clusters, arms, or separations in this plot suggest distinct biological
subgroups exist in the data without labels. If the data separates into 3 to 4
visible regions, this is strong early evidence that unsupervised clustering may recover
the known breast cancer molecular subtypes: Luminal A, Luminal B, HER2-enriched, and Triple-Negative.

**Original feature space:**
Plotting on two arbitrary genes tells us almost nothing useful. Any two genes out of
17,814 carry only a tiny fraction of the total information and produce a noisy,
structureless scatter. PCA solves this by finding the two directions in the 5,000-dimensional
gene space that capture the most total variation, making structure visible that would
be completely invisible in any pair of raw gene plots.

---
## 8. How Many PCs to Retain for Downstream Clustering?

In [ ]:
# Find PC counts at key variance thresholds
print('Variance threshold  |  PCs needed')
print('-' * 35)
for t in [0.50, 0.80, 0.90]:
    n = int(np.argmax(cumulative_var >= t) + 1)
    print(f'  {t*100:.0f}%              |  {n}')

# Use 50% threshold for clustering - compact but signal-rich
N_PCS = int(np.argmax(cumulative_var >= 0.50) + 1)
print(f'\nSelected: {N_PCS} PCs (50% variance threshold)')
print('Rationale: retains dominant biological signal while keeping dimensionality manageable.')
print('Consistent with common practice in cancer genomics literature.')

---
## 9. Save Reduced Representation for Clustering

We save the PCA-reduced matrix so Notebooks 4 (K-Means) and 5 (Hierarchical Clustering)
can load it directly without re-running PCA.

In [ ]:
X_reduced = X_pca[:, :N_PCS]

np.save(DATA_DIR / 'X_pca_reduced.npy', X_reduced)

print('Saved:')
print(f'  data/X_pca_reduced.npy  shape {X_reduced.shape}')
print(f'  ({X_reduced.shape[0]} samples x {X_reduced.shape[1]} principal components)')
print()
print('This file is the input for Notebook 4 (K-Means) and Notebook 5 (Hierarchical Clustering).')

---
## 10. Summary

| Property | Value |
|---|---|
| Input matrix | BC-TCGA Tumor, 529 samples x 5,000 genes |
| Method | PCA (sklearn) + SVD (numpy) |
| Variance in PC1 | see cell output above |
| Variance in PC2 | see cell output above |
| PCs for 50% variance | see cell output above |
| PCs for 80% variance | see cell output above |
| PCs retained for clustering | 50% variance threshold |
| Output file | `data/X_pca_reduced.npy` |

### Key Takeaways

1. The scree plot shows a steep initial drop followed by a long flat tail, confirming that
   a small number of PCs capture the dominant biological variation across all 529 tumors.

2. The U matrix positions each tumor sample in PC space. Samples close together
   share similar gene expression profiles and likely belong to the same molecular subtype.

3. The V* matrix reveals which genes drive each PC. Genes with high loadings on early
   PCs are likely biologically meaningful markers related to subtype-defining pathways
   such as estrogen receptor signaling, HER2 amplification, or proliferation.

4. The PC1/PC2 scatter plot gives our first visual hint of whether natural clusters
   exist without any labels, setting the stage for formal clustering in Notebooks 4 and 5.

5. Plotting in PC space reveals far more structure than plotting on any two original
   genes, demonstrating exactly why dimensionality reduction is the right first step
   for this kind of high-dimensional data.